In [ ]:
import os
import re
import json
import ast
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM


In [ ]:
DATA_DIR = Path("./Data")

INPUT_CSV = DATA_DIR / "Phi3_Responses_Wikipedia_Only.csv"

FAISS_PATH = Path("WIKI_resource/wiki_retrieval_output/wiki_passages.faiss")
PASSAGES_PATH = Path("WIKI_resource/wiki_retrieval_output/wiki_passages.csv")

LOCAL_JUDGE_DIR = Path("fever_deberta_judge")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

CLAIM_OUTPUT_CSV = OUTPUT_DIR / "claims_with_faiss_and_judgments.csv"
ANSWER_OUTPUT_CSV = OUTPUT_DIR / "answer_hallucination_results.csv"

CLAIM_JUDGE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

TOP_K = 3

In [ ]:
print("cwd:    ", Path.cwd())
print("1 up:   ", Path("..").resolve())
print("2 up:   ", Path("../..").resolve())
print("3 up:   ", Path("../../..").resolve())

print("\nData:", Path("../../../Data").resolve())

In [ ]:
df = pd.read_csv(INPUT_CSV)

required = {"Question", "Phi3_Answer"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if "Question_ID" not in df.columns:
    df = df.reset_index(drop=True)
    df["Question_ID"] = np.arange(1, len(df) + 1)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head())


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    CLAIM_JUDGE_MODEL,
    trust_remote_code=True
)

llm = AutoModelForCausalLM.from_pretrained(
    CLAIM_JUDGE_MODEL,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)
llm.eval()

print("Loaded:", CLAIM_JUDGE_MODEL)
print("Device:", next(llm.parameters()).device)


In [ ]:
CLAIM_SYSTEM_PROMPT = """
You are a factual claim extraction assistant.

Your task is to decompose an answer into atomic, independently verifiable factual claims.

Rules:
1. Each claim must contain exactly ONE factual proposition.
2. Split compound statements into separate claims.
3. Preserve important entities, dates, quantities, locations, and relationships.
4. Resolve simple pronouns when possible so each claim can stand alone.
5. Do NOT include opinions, advice, hedging, greetings, refusals, or non-factual filler.
6. Do NOT judge whether the claims are true or false.
7. Do NOT add new information.
8. If the answer contains no verifiable factual claim, return an empty list.
9. Return valid JSON only, exactly as: {"claims": ["claim 1", "claim 2"]}
"""


def parse_claims(text):
    if not isinstance(text, str):
        return []
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    for candidate in [text] + re.findall(r"\{.*?\}", text, flags=re.S):
        try:
            obj = json.loads(candidate)
            if isinstance(obj, dict) and isinstance(obj.get("claims"), list):
                return [str(x).strip() for x in obj["claims"] if str(x).strip()]
        except Exception:
            pass

    try:
        obj = ast.literal_eval(text)
        if isinstance(obj, dict) and isinstance(obj.get("claims"), list):
            return [str(x).strip() for x in obj["claims"] if str(x).strip()]
    except Exception:
        pass

    claims = []
    for line in text.splitlines():
        line = re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", line).strip()
        if line and not line.startswith("{") and not line.startswith("}"):
            claims.append(line)
    return claims


@torch.inference_mode()
def extract_claims(question, answer, max_new_tokens=300):
    messages = [
        {"role": "system", "content": CLAIM_SYSTEM_PROMPT.strip()},
        {"role": "user", "content": f"Question:\n{question}\n\nGenerated Answer:\n{answer}\n\nExtract the atomic factual claims."}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3500).to(llm.device)
    outputs = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return parse_claims(raw), raw


In [ ]:
claim_rows = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting claims"):
    question = str(row["Question"])
    answer = str(row["Phi3_Answer"])

    try:
        claims, raw = extract_claims(question, answer)
    except Exception as e:
        print(f"Claim extraction failed for Question_ID={row['Question_ID']}: {e}")
        claims, raw = [], f"ERROR: {e}"

    for claim_id, claim in enumerate(claims, start=1):
        claim_rows.append({
            "Question_ID": row["Question_ID"],
            "Question": row["Question"],
            "Phi3_Answer": row["Phi3_Answer"],
            "Source": row.get("Source", ""),
            "Claim_ID": claim_id,
            "Atomic_Claim": claim,
            "Claim_Extraction_Raw": raw,
        })

claims_df = pd.DataFrame(claim_rows)
print("Extracted claims:", len(claims_df))
display(claims_df.head(20))


In [ ]:
if len(claims_df) and "Source" in claims_df.columns:
    source_text = claims_df["Source"].fillna("").astype(str)
    wiki_mask = source_text.str.contains(r"wikipedia\.org", case=False, regex=True)

    if wiki_mask.any():
        claims_df = claims_df[wiki_mask].copy().reset_index(drop=True)
        print("Claims after Wikipedia-only filter:", len(claims_df))
    else:
        print("No Wikipedia URLs detected in Source; filter not applied.")


In [ ]:
if not FAISS_PATH.exists():
    raise FileNotFoundError(f"FAISS index not found: {FAISS_PATH}")
if not PASSAGES_PATH.exists():
    raise FileNotFoundError(f"Passage CSV not found: {PASSAGES_PATH}")

index = faiss.read_index(str(FAISS_PATH))
passages_df = pd.read_csv(PASSAGES_PATH)

if index.ntotal != len(passages_df):
    raise ValueError(
        f"FAISS/passages mismatch: index={index.ntotal}, passages={len(passages_df)}"
    )

retriever = SentenceTransformer(EMBEDDING_MODEL)
embedding_dim = retriever.get_sentence_embedding_dimension()

if index.d != embedding_dim:
    raise ValueError(f"Embedding dimension mismatch: FAISS={index.d}, model={embedding_dim}")

print("FAISS vectors:", index.ntotal)
print("FAISS dimension:", index.d)
print("Embedding model:", EMBEDDING_MODEL)


In [ ]:
def search_claim(claim, k=TOP_K):
    q = retriever.encode(
        [str(claim).strip()],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(q, k)
    valid = indices[0] >= 0
    result_indices = indices[0][valid]
    result_scores = scores[0][valid]

    results = passages_df.iloc[result_indices].copy().reset_index(drop=True)
    results["score"] = result_scores
    return results


retrieval_rows = []
for _, row in tqdm(claims_df.iterrows(), total=len(claims_df), desc="FAISS retrieval"):
    output = row.to_dict()
    claim = str(row.get("Atomic_Claim", "")).strip()

    if not claim:
        output["Retrieval_Status"] = "SKIPPED_EMPTY_CLAIM"
        results = pd.DataFrame()
    else:
        try:
            results = search_claim(claim, TOP_K)
            output["Retrieval_Status"] = "SUCCESS"
        except Exception as e:
            results = pd.DataFrame()
            output["Retrieval_Status"] = "ERROR"
            output["Retrieval_Error"] = str(e)

    for rank in range(1, TOP_K + 1):
        if rank <= len(results):
            ev = results.iloc[rank - 1]
            output[f"Evidence_{rank}"] = ev.get("passage_text", "")
            output[f"Evidence_{rank}_Score"] = ev.get("score", np.nan)
            output[f"Evidence_{rank}_Page"] = ev.get("page_title", "")
            output[f"Evidence_{rank}_URL"] = ev.get("source_url", "")
        else:
            output[f"Evidence_{rank}"] = ""
            output[f"Evidence_{rank}_Score"] = np.nan
            output[f"Evidence_{rank}_Page"] = ""
            output[f"Evidence_{rank}_URL"] = ""

    retrieval_rows.append(output)

retrieval_df = pd.DataFrame(retrieval_rows)
print(retrieval_df["Retrieval_Status"].value_counts(dropna=False))
display(retrieval_df.head())


In [ ]:
JUDGE_SYSTEM_PROMPT = """
You are an evidence-based factuality judge.

Your task is to determine whether a factual claim is supported by the provided evidence.
You must use ONLY the provided evidence. Do NOT use your own background knowledge.

Labels:
SUPPORTED: The evidence directly supports the important factual content of the claim.
UNSUPPORTED: The evidence does not establish the claim, contradicts it, is irrelevant, or is insufficient.

Rules:
1. Judge the whole atomic claim.
2. Do not assume missing facts.
3. High topical similarity does NOT automatically mean support.
4. If evidence discusses the same topic but does not establish the claimed fact, return UNSUPPORTED.
5. If no usable evidence is provided, return UNSUPPORTED.
6. Return valid JSON only.

Output exactly:
{"label": "SUPPORTED or UNSUPPORTED", "reason": "brief evidence-based explanation"}
"""


def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if not x or x.lower().startswith("error:"):
        return ""
    return x


def combine_evidence(row):
    pieces = []
    for i in range(1, TOP_K + 1):
        text = clean_text(row.get(f"Evidence_{i}", ""))
        page = clean_text(row.get(f"Evidence_{i}_Page", ""))
        if text:
            header = f"[Evidence {i} | Wikipedia page: {page}]" if page else f"[Evidence {i}]"
            pieces.append(f"{header}\n{text}")
    return "\n\n".join(pieces) if pieces else "[NO USABLE RETRIEVED EVIDENCE]"


def parse_judgment(text):
    if not isinstance(text, str):
        return "UNSUPPORTED", "Judge returned no parseable output."

    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.I)
    candidates = [text] + re.findall(r"\{.*?\}", text, flags=re.S)
    for candidate in candidates:
        try:
            obj = json.loads(candidate)
            label = str(obj.get("label", "")).strip().upper()
            reason = str(obj.get("reason", "")).strip()
            if label in {"SUPPORTED", "UNSUPPORTED"}:
                return label, reason
        except Exception:
            pass

    upper = text.upper()
    if "UNSUPPORTED" in upper:
        return "UNSUPPORTED", text
    if "SUPPORTED" in upper:
        return "SUPPORTED", text
    return "UNSUPPORTED", f"Unparseable judge output: {text}"


@torch.inference_mode()
def judge_claim_prompt(claim, evidence_text, max_new_tokens=160):
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT.strip()},
        {"role": "user", "content": f"Atomic Claim:\n{claim}\n\nRetrieved Evidence:\n{evidence_text}\n\nDetermine whether the claim is SUPPORTED or UNSUPPORTED."}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3500).to(llm.device)
    outputs = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
    label, reason = parse_judgment(raw)
    return label, reason, raw


In [ ]:
judged_rows = []

for _, row in tqdm(retrieval_df.iterrows(), total=len(retrieval_df), desc="Judging claims"):
    output = row.to_dict()
    evidence = combine_evidence(row)
    claim = str(row["Atomic_Claim"])

    try:
        label, reason, raw = judge_claim_prompt(claim, evidence)
    except Exception as e:
        label = "UNSUPPORTED"
        reason = f"JUDGE ERROR: {e}"
        raw = f"ERROR: {e}"

    output["Combined_Evidence"] = evidence
    output["Prompt_Judge_Label"] = label
    output["Prompt_Judge_Reason"] = reason
    output["Prompt_Judge_Raw"] = raw
    judged_rows.append(output)

judged_df = pd.DataFrame(judged_rows)
judged_df.to_csv(CLAIM_OUTPUT_CSV, index=False)

print("Saved claim-level results:", CLAIM_OUTPUT_CSV)
display(judged_df[["Question_ID", "Claim_ID", "Atomic_Claim", "Prompt_Judge_Label", "Prompt_Judge_Reason"]].head(20))


In [ ]:
def aggregate_answer(group):
    labels = group["Prompt_Judge_Label"].astype(str).str.upper().tolist()
    n_claims = len(labels)
    n_supported = sum(x == "SUPPORTED" for x in labels)
    n_unsupported = sum(x == "UNSUPPORTED" for x in labels)
    unsupported_ratio = n_unsupported / n_claims if n_claims else np.nan

    hallucination = "YES" if n_unsupported > 0 else "NO"

    return pd.Series({
        "Num_Claims": n_claims,
        "Num_Supported": n_supported,
        "Num_Unsupported": n_unsupported,
        "Unsupported_Ratio": unsupported_ratio,
        "Hallucination": hallucination,
    })


if len(judged_df):
    answer_results = (
        judged_df.groupby("Question_ID", dropna=False)
        .apply(aggregate_answer, include_groups=False)
        .reset_index()
    )

    answer_meta = df[["Question_ID", "Question", "Phi3_Answer"]].drop_duplicates("Question_ID")
    answer_results = answer_meta.merge(answer_results, on="Question_ID", how="right")
else:
    answer_results = pd.DataFrame(columns=[
        "Question_ID", "Question", "Phi3_Answer", "Num_Claims", "Num_Supported",
        "Num_Unsupported", "Unsupported_Ratio", "Hallucination"
    ])

answer_results.to_csv(ANSWER_OUTPUT_CSV, index=False)
print("Saved answer-level results:", ANSWER_OUTPUT_CSV)
display(answer_results.head(20))


In [ ]:
summary = {
    "num_answers_with_claims": len(answer_results),
    "num_claims": len(judged_df),
    "supported_claims": int((judged_df.get("Prompt_Judge_Label", pd.Series(dtype=str)) == "SUPPORTED").sum()),
    "unsupported_claims": int((judged_df.get("Prompt_Judge_Label", pd.Series(dtype=str)) == "UNSUPPORTED").sum()),
    "hallucinated_answers": int((answer_results.get("Hallucination", pd.Series(dtype=str)) == "YES").sum()),
    "non_hallucinated_answers": int((answer_results.get("Hallucination", pd.Series(dtype=str)) == "NO").sum()),
}

if len(judged_df):
    summary["claim_support_rate"] = float((judged_df["Prompt_Judge_Label"] == "SUPPORTED").mean())
if len(answer_results):
    summary["answer_hallucination_rate"] = float((answer_results["Hallucination"] == "YES").mean())

for k, v in summary.items():
    print(f"{k}: {v}")


In [ ]:
import pandas as pd
from openai import OpenAI
import time


client = OpenAI()


input_file = "./outputs/answer_hallucination_results.csv"
output_file = "./outputs/openai_direct_judge_results.csv"

df = pd.read_csv(input_file)



def openai_hallucination_judge(question, answer):

    prompt = f"""
You are a hallucination detection judge.

Your task is to determine whether the provided answer contains factual
hallucination with respect to the question.

A hallucination occurs when the answer:
- contains factually incorrect information,
- invents unsupported facts,
- incorrectly assumes something in the question is true,
- gives a misleading factual answer,
- or confidently provides information that is not actually true.

An answer is NOT a hallucination if:
- it is factually correct,
- it appropriately expresses uncertainty,
- or it correctly points out that the premise of the question is false.

Evaluate ONLY the Question and the Answer below.

Do not use any provided reference answer or dataset label.

Question:
{question}

Answer:
{answer}

Return exactly one of the following:

YES
NO

YES = the answer contains hallucination.
NO = the answer does not contain hallucination.
"""

    response = client.responses.create(
        model="gpt-5-mini",
        input=prompt
    )

    result = response.output_text.strip().upper()

    if result.startswith("YES"):
        return "YES"

    elif result.startswith("NO"):
        return "NO"

    else:
        return "UNKNOWN"


openai_results = []

for i, row in df.iterrows():

    question = row["Question"]
    answer = row["Phi3_Answer"]

    try:

        result = openai_hallucination_judge(
            question,
            answer
        )

        openai_results.append(result)

        print(
            f"{i+1}/{len(df)} | "
            f"Question_ID={row['Question_ID']} | "
            f"OpenAI Judge={result}"
        )

    except Exception as e:

        print(
            f"Error at row {i}: {e}"
        )

        openai_results.append("ERROR")

    time.sleep(0.2)



df["OpenAI_Judge"] = openai_results

df.to_csv(
    output_file,
    index=False
)

print("\nFinished.")
print("Saved to:", output_file)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_true = df["Hallucination"].str.strip().str.upper()

y_pred = df["OpenAI_Judge"].str.strip().str.upper()


accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true, y_pred,
    pos_label="YES"
)

recall = recall_score(
    y_true, y_pred,
    pos_label="YES"
)

f1 = f1_score(
    y_true, y_pred,
    pos_label="YES"
)

print("===== OpenAI Judge Performance =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\n===== Classification Report =====")
print(
    classification_report(
        y_true,
        y_pred,
        labels=["YES", "NO"]
    )
)

print("\n===== Confusion Matrix =====")
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=["YES", "NO"]
)

print(cm)